# PaliGemma 2: Türkçe fişlerde 224 / 448 ve İngilizce / Türkçe soru
**Tuğçe Karakuş · Araştırma ve portföy projesi**

Bu defter GitHub için hazırlanan `paligemma2-turkish-receipts` projesini çalıştırır. Hedef **50 farklı gerçek fiş**, asgari 30. Her fiş dört kez okunur: iki model × iki soru dili (İngilizce, Türkçe).

Önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin. [mix-224](https://huggingface.co/google/paligemma2-3b-mix-224) ve [mix-448](https://huggingface.co/google/paligemma2-3b-mix-448) sayfalarında Google erişim koşullarını kendi hesabınızla kabul edin. Giriş token'ını paylaşmayın.

Gerekenler: 30–50 fiş görüntüsü (tek ZIP olabilir) ve doldurulmuş `manifest.csv`. Ayrıntılı veri yönergesi proje içindeki `docs/VERI_VE_DENEY.md` dosyasında. Test cevaplarını gördükten sonra soruyu değiştirmeyin.

In [ ]:
%pip -q install transformers==4.57.1 accelerate==1.11.0 huggingface_hub==0.36.0 sentencepiece==0.2.1 'Pillow>=10.4,<13' 'numpy>=1.26,<3' 'matplotlib>=3.8,<4'

## 1. Projeyi GitHub’dan alın
Bu hücre projeyi indirir ve temel kontrolleri çalıştırır. README ve veri yönergesini gözden geçirin.

In [ ]:
import sys, json, zipfile, io, subprocess
from pathlib import Path
from google.colab import files
from IPython.display import display, Image as DisplayImage
BASE = Path('/content/paligemma2_portfolio')
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / 'paligemma2-turkish-receipts'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/karakustugce/paligemma2-turkish-receipts.git', str(PROJECT)], check=True)
assert (PROJECT/'pg2receipts/run.py').is_file(), 'Proje indirilemedi.'
sys.path.insert(0, str(PROJECT))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=PROJECT, check=True)
print('Proje ve temel kontroller hazır.')

## 2. Model erişimi ve GPU
CUDA GPU bulunmuyorsa burada durun ve çalışma zamanını değiştirin. Modeller sırayla ve FP16 ile yüklenir. GPU belleği veya erişim sorunu olduğunda koşu durur; bunu başarısız model cevabı olarak saymayın.

In [ ]:
import torch
from huggingface_hub import notebook_login
assert torch.cuda.is_available(), 'GPU çalışma zamanı seçilmedi.'
print('GPU:', torch.cuda.get_device_name(0))
print('Bellek (GB):', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
notebook_login()

## 3. Referans toplamlarını doğrulayın
Kaynak örneklemde 49 gerçek görüntü var; referans tutar dosyası özel tutuluyor. Aşağıdaki hücre bu CSV dosyasını yüklemenizi ister. Her görüntüyü kontrol edin; doğru öneride Enter ile onaylayın, yanlışsa doğru toplamı girin. Onaylar kaydedilir. Fișlerden biri belirsizse kaynağı düzeltmeden koşuya geçmeyin.

Kendi fişlerinizi kullanmak için aşağıda `USE_PUBLIC_SAMPLE = False` yapın ve manifest.csv ile görüntüleri yükleyin. Kaynak örneklem hakkında ayrıntılar `docs/KAYNAK_VERI.md` içindedir.


In [ ]:
import csv
from PIL import Image, ImageOps
from pg2receipts.core import load_manifest, parse_amount
DATA = BASE/'private_data'; IMAGES = DATA/'images'
IMAGES.mkdir(parents=True, exist_ok=True)
USE_PUBLIC_SAMPLE = True
MANIFEST = DATA/'manifest.csv'
if USE_PUBLIC_SAMPLE:
    print('Özel kaynak referans CSV dosyasını yükleyin; dosya açık depoda bulunmuyor.')
    reference_upload = files.upload()
    assert len(reference_upload) == 1 and next(iter(reference_upload)).lower().endswith('.csv')
    draft_input = DATA/'source_reference_private.csv'
    draft_input.write_bytes(next(iter(reference_upload.values())))
    subprocess.run([sys.executable, '-m', 'pg2receipts.prepare', '--out', str(DATA), '--draft-manifest', str(draft_input)], cwd=PROJECT, check=True)
    source_csv = MANIFEST if MANIFEST.exists() else DATA/'manifest_draft.csv'
    with source_csv.open(encoding='utf-8', newline='') as f:
        rows=list(csv.DictReader(f))
    for row in rows:
        if row.get('annotation_status') == 'human_verified': continue
        with Image.open(IMAGES/row['image_path']) as raw:
            preview=ImageOps.exif_transpose(raw).convert('RGB')
            preview.thumbnail((600,1100)); display(preview)
        answer=input(f"{row['doc_id']} — önerilen toplam {row['true_total']}; doğruysa Enter, değilse doğru toplam: ").strip()
        value=parse_amount(answer or row['true_total'])
        if value is None: raise ValueError('Tutar belirsiz. Geçerli iki ondalıklı tutar girin.')
        row['true_total']=str(value); row['annotation_status']='human_verified'
        with MANIFEST.open('w', encoding='utf-8', newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
else:
    uploaded = files.upload()
    manifest_bytes = None; image_names = set()
    def save_image(name, data):
        filename = Path(name).name
        assert filename not in image_names, 'Tekrarlanan görüntü adı: '+filename
        image_names.add(filename)
        (IMAGES/filename).write_bytes(data)
    for name, data in uploaded.items():
        suffix=Path(name).suffix.lower()
        if suffix == '.csv':
            assert manifest_bytes is None, 'Tek bir manifest CSV yükleyin.'
            manifest_bytes=data
        elif suffix == '.zip':
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for item in z.infolist():
                    assert '..' not in Path(item.filename).parts and not Path(item.filename).is_absolute()
                    if not item.is_dir() and Path(item.filename).suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
                        save_image(item.filename, z.read(item))
        elif suffix in {'.jpg','.jpeg','.png','.webp'}:
            save_image(name, data)
        else:
            raise ValueError('Desteklenmeyen dosya: '+name)
    assert manifest_bytes is not None, 'Doğrulanmış manifest.csv dosyasını da yükleyin.'
    MANIFEST = DATA/'manifest.csv'; MANIFEST.write_bytes(manifest_bytes)
docs = load_manifest(MANIFEST, IMAGES)
from collections import Counter
print('Geçerli farklı belgeler:', len(docs))
assert input('Referansları kontrol ettim ve test setini donduruyorum. EVET: ').strip().upper() == 'EVET'


## 4. Aynı belgeleri iki modele okutun
Model indirme ilk çalıştırmada zaman alır. Gerçek tutarlar modele verilmez. Ham cevaplar her belgeden sonra kaydedilir. Kesilen koşuyu aynı görüntüler, cevaplar ve ortamla yeniden çalıştırmak kaldığı yerden devam eder. Soru, veri veya yazılım sürümü değişirse yeni koşu gerekir.

Her fiş aynı soruyla hem İngilizce hem Türkçe sorulur; böylece soru dilinin etkisi de ölçülür. Model karşılaştırması iki **mix kontrol noktası** arasındadır; makalenin DocVQA sonuçlarının tekrarı veya yalnız çözünürlüğü değiştiren nedensel bir deney değildir.

In [ ]:
OUT = BASE/'real_run'
command=[sys.executable, '-m', 'pg2receipts.run', '--manifest', str(MANIFEST), '--images', str(IMAGES), '--out', str(OUT)]
if (OUT/'run_config.json').exists(): command.append('--resume')
subprocess.run(command, cwd=PROJECT, check=True)

## 5. Analiz ve görseller
Başarı, toplam tutarın tam eşleşmesidir. Açıklamalı/çoklu tutarlı cevaplar sabit ayrıştırma kuralı altında başarısız sayılır; biçim hataları ayrıca gösterilir. Üretim süreleri ve GPU tahsisli bellek ölçülür; insan kontrol süresi ve parasal yatırım getirisi ölçülmez.

Doğrulukta Wilson aralıkları, farkta eşleştirilmiş bootstrap ve keşifsel McNemar testi kullanılır. Küçük örneklem ve benzer işletme şablonları nedeniyle bulguları bütün Türkçe belgelere genellemeyin. Ekonomik grafikteki **insan süreleri varsayımsaldır**.

In [ ]:
subprocess.run([sys.executable, '-m', 'pg2receipts.analyze', '--run', str(OUT)], cwd=PROJECT, check=True)
summary=json.loads((OUT/'summary.json').read_text())
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(OUT/'accuracy_and_resources.png')))
display(DisplayImage(filename=str(OUT/'economic_scenario.png')))
print((OUT/'report.md').read_text())

## 6. Sonuç paketini indirin
Paket ham yanıtlar, referans tutarlar, sürümler, özet, rapor ve grafikleri içerir; fiş görüntüleri veya giriş token'ı içermez. Paylaşmadan önce inceleyin. Kamuya açık GitHub deposuna önce kodu, sonra gözden geçirilmiş toplu sonuçları ekleyin. Görselleri ve gerçek sayılarını makaleye aktarın; bütün boşlukları bu koşunun sonuçlarıyla doldurun.

In [ ]:
archive=BASE/'PaliGemma_2_Gercek_Deney_Sonuclari.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): z.write(path, arcname=path.name)
files.download(str(archive))